# AlzheimerCNN Ablation Study

This notebook is the ablation study for the `AlzheimerCNN` model contribution to PyHealth.

**Reference paper:** Liu et al. *"On the design of convolutional neural networks for automatic detection of Alzheimer's disease."* ML4H @ NeurIPS 2019. https://arxiv.org/abs/1911.03740

## Experimental Setup

We train five model configurations on the `Falah/Alzheimer_MRI` dataset and compare validation accuracy and macro-F1. All models share the same training recipe:

- **Optimizer:** AdamW, lr=1e-4, weight_decay=1e-5
- **Epochs:** 50
- **Batch size:** 32
- **Train/Val/Test split:** 60% / 20% / 20%
- **Loss:** Cross-entropy (4-class classification)

## Configurations Tested

| # | Model | Variation | Hypothesis |
|---|-------|-----------|------------|
| 1 | `AlzheimerCNN` | `init_channels=32` (baseline) | Reference point |
| 2 | `AlzheimerCNN` | `init_channels=16` | Smaller capacity may underfit |
| 3 | `AlzheimerCNN` | `init_channels=64` | Larger capacity may overfit on 5k samples |
| 4 | `AlzheimerCNNNormVariant` | `norm_type="group"` | GroupNorm may match InstanceNorm at small batch sizes |
| 5 | `AlzheimerCNNNormVariant` | `norm_type="layer"` | LayerNorm may match InstanceNorm at small batch sizes |
| 6 | `AlzheimerCNNViT` | CNN + Transformer hybrid | Self-attention may model global context better, but on small set may not be great |


## 1. Setup


In [1]:
import torch
import pandas as pd

from pyhealth.datasets import split_by_sample, get_dataloader
from pyhealth.trainer import Trainer
from pyhealth.models import (
    AlzheimerCNN,
    AlzheimerCNNNormVariant,
    AlzheimerCNNViT,
)

# ── Auto-detect device ──────────────────────────────────────────────────
DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"

print(f"Using device: {DEVICE}")

METRICS = ["accuracy", "f1_macro", "balanced_accuracy"]
EPOCHS = 50
BATCH_SIZE = 32
SEED = 42

torch.manual_seed(SEED)


C:\Users\spicy\Downloads\testfinal\PyHealth\pyhealth\trainer.py:12: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import trange


Using device: cpu


## 2. Load Dataset

Uses the `AlzheimersDataset` PyHealth-compatible wrapper around `Falah/Alzheimer_MRI` (4-class severity classification: NonDemented, VeryMildDemented, MildDemented, ModerateDemented).

The dataset class auto-downloads from HuggingFace and saves a `alzheimers-metadata.csv` for PyHealth's BaseDataset to consume.


In [2]:
from typing import Any, Dict, List
from pyhealth.tasks.base_task import BaseTask


class AlzheimersClassification(BaseTask):
    """Task for classifying Alzheimer's disease from brain images."""

    task_name: str = "AlzheimersClassification"
    input_schema: Dict = {"image": ("image", {"mode": "L", "image_size": 128})}
    output_schema: Dict[str, str] = {"label": "multiclass"}

    def __call__(self, patient: Any) -> List[Dict[str, Any]]:
        events = patient.get_events(event_type="alzheimers")
        assert len(events) == 1, f"Expected 1 image, got {len(events)}"
        event = events[0]
        return [{"image": event.path, "label": event.label}]

In [3]:
import logging
import os
from pathlib import Path
from typing import Optional

import pandas as pd
from datasets import load_dataset
from pyhealth.datasets.base_dataset import BaseDataset

logger = logging.getLogger(__name__)


class AlzheimersDataset(BaseDataset):
    """PyHealth-compatible Alzheimer's dataset (auto-download + process)."""

    def __init__(
        self,
        root: str,
        dataset_name: Optional[str] = None,
        config_path: Optional[str] = None,
        cache_dir: Optional[str] = None,
        num_workers: int = 1,
        dev: bool = False,
        hf_dataset_name: str = "Falah/Alzheimer_MRI",
    ) -> None:
        self.hf_dataset_name = hf_dataset_name

        try:
            base_path = Path(__file__).parent
        except NameError:
            base_path = Path.cwd()

        if config_path is None:
            config_path = base_path / "configs" / "alzheimers.yaml"

        metadata_path = os.path.join(root, "alzheimers-metadata.csv")

        if not os.path.exists(metadata_path):
            self.prepare_metadata(root)

        super().__init__(
            root=root,
            tables=["alzheimers"],
            dataset_name=dataset_name or "alzheimers",
            config_path=config_path,
            cache_dir=cache_dir,
            num_workers=num_workers,
            dev=dev,
        )

    def prepare_metadata(self, root: str) -> None:
        os.makedirs(root, exist_ok=True)
        image_dir = os.path.join(root, "images")
        os.makedirs(image_dir, exist_ok=True)
        metadata_path = os.path.join(root, "alzheimers-metadata.csv")

        ds = load_dataset(self.hf_dataset_name, split="train")
        records = []
        for i, item in enumerate(ds):
            img_path = os.path.join(image_dir, f"{i}.png")
            if not os.path.isfile(img_path):
                item["image"].save(img_path)
            records.append({
                "patient_id": str(i),
                "path": img_path,
                "label": str(item["label"]),
            })
        pd.DataFrame(records).to_csv(metadata_path, index=False)

In [4]:
# Instantiate dataset and create train/val/test splits
dataset = AlzheimersDataset(root="Downloads")
dataset_samples = dataset.set_task(AlzheimersClassification(), num_workers=1)

train_samples, val_samples, test_samples = split_by_sample(
    dataset_samples, [0.6, 0.2, 0.2]
)

train_loader = get_dataloader(train_samples, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = get_dataloader(val_samples,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = get_dataloader(test_samples,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

Initializing alzheimers dataset from Downloads (dev mode: False)
No cache_dir provided. Using default cache dir: C:\Users\spicy\AppData\Local\pyhealth\pyhealth\Cache\5e3d2e44-3165-596e-89d3-ed772a0ed564
Setting task AlzheimersClassification for alzheimers base dataset...
Task cache paths: task_df=C:\Users\spicy\AppData\Local\pyhealth\pyhealth\Cache\5e3d2e44-3165-596e-89d3-ed772a0ed564\tasks\AlzheimersClassification_7e906453-ea6b-5cc3-8506-3e00427d79ba\task_df.ld, samples=C:\Users\spicy\AppData\Local\pyhealth\pyhealth\Cache\5e3d2e44-3165-596e-89d3-ed772a0ed564\tasks\AlzheimersClassification_7e906453-ea6b-5cc3-8506-3e00427d79ba\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Found cached processed samples at C:\Users\spicy\AppData\Local\pyhealth\pyhealth\Cache\5e3d2e44-3165-596e-89d3-ed772a0ed564\tasks\AlzheimersClassification_7e906453-ea6b-5cc3-8506-3e00427d79ba\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld, skipping processing.
Train: 3072 | Val: 1024 | Test: 1024


## 3. Training Helper

A single function that trains any model with the shared recipe and returns final validation metrics. This keeps every experiment cell short and ensures all configurations are compared fairly.


In [5]:
import time

def train_and_evaluate(model, name: str) -> dict:
    """Train a model and return its final validation metrics.

    Args:
        model: An instantiated PyHealth model inheriting from BaseModel.
        name: Human-readable label for the configuration.

    Returns:
        Dict with keys 'name', 'accuracy', 'f1_macro', 'balanced_accuracy', 'loss', 'params'.
    """
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")

    trainer = Trainer(model=model, metrics=METRICS, device=DEVICE)

    t0 = time.perf_counter()
    trainer.train(
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        epochs=EPOCHS,
        optimizer_class=torch.optim.AdamW,
        optimizer_params={"lr": 1e-4, "weight_decay": 1e-5},
    )
    train_time = time.perf_counter() - t0

    scores = trainer.evaluate(val_loader)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        "name": name,
        "accuracy": scores.get("accuracy", float("nan")),
        "f1_macro": scores.get("f1_macro", float("nan")),
        "balanced_accuracy": scores.get("balanced_accuracy", float("nan")),
        "loss": scores.get("loss", float("nan")),
        "params": num_params,
        "train_time_seconds": round(train_time, 1),
    }


# Collect all results in this list for the final summary
results = []


## 4. Experiments


### Experiment 1: Baseline (`init_channels=32`)


In [6]:
model = AlzheimerCNN(
    dataset=train_samples,
    init_channels=32,
    classifier_hidden_dim=128,
    dropout=0.5,
)
results.append(train_and_evaluate(model, "CNN (init_channels=32, baseline)"))



Training: CNN (init_channels=32, baseline)
AlzheimerCNN(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kerne

Epoch 0 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-0, step-96 ---
loss: 1.1702


Evaluation: 100%|██████████| 32/32 [00:02<00:00, 11.83it/s]

--- Eval epoch-0, step-96 ---
accuracy: 0.5078
f1_macro: 0.1684
balanced_accuracy: 0.2500
loss: 1.0494



...

Epoch 49 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-49, step-4800 ---
loss: 0.4850


Evaluation: 100%|██████████| 32/32 [00:02<00:00, 12.26it/s]

--- Eval epoch-49, step-4800 ---
accuracy: 0.7324
f1_macro: 0.5123
balanced_accuracy: 0.4980
loss: 0.6305



Evaluation: 100%|██████████| 32/32 [00:02<00:00, 12.22it/s]


### Experiment 2: Smaller capacity (`init_channels=16`)


In [7]:
model = AlzheimerCNN(
    dataset=train_samples,
    init_channels=16,
    classifier_hidden_dim=128,
    dropout=0.5,
)
results.append(train_and_evaluate(model, "CNN (init_channels=16)"))



Training: CNN (init_channels=16)
AlzheimerCNN(
  (block1): Sequential(
    (0): Conv2d(1, 16, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, st

Epoch 0 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-0, step-96 ---
loss: 1.2238


Evaluation: 100%|██████████| 32/32 [00:01<00:00, 29.52it/s]

--- Eval epoch-0, step-96 ---
accuracy: 0.5078
f1_macro: 0.1684
balanced_accuracy: 0.2500
loss: 1.0845



...

Epoch 49 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-49, step-4800 ---
loss: 0.8174


Evaluation: 100%|██████████| 32/32 [00:01<00:00, 28.66it/s]

--- Eval epoch-49, step-4800 ---
accuracy: 0.6133
f1_macro: 0.3255
balanced_accuracy: 0.3537
loss: 0.8018



Evaluation: 100%|██████████| 32/32 [00:01<00:00, 29.34it/s]


### Experiment 3: Larger capacity (`init_channels=64`)


In [8]:
model = AlzheimerCNN(
    dataset=train_samples,
    init_channels=64,
    classifier_hidden_dim=128,
    dropout=0.5,
)
results.append(train_and_evaluate(model, "CNN (init_channels=64)"))



Training: CNN (init_channels=64)
AlzheimerCNN(
  (block1): Sequential(
    (0): Conv2d(1, 64, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(128, 256, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(256, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=

Epoch 0 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-0, step-96 ---
loss: 1.0940


Evaluation: 100%|██████████| 32/32 [00:07<00:00,  4.29it/s]

--- Eval epoch-0, step-96 ---
accuracy: 0.5078
f1_macro: 0.1684
balanced_accuracy: 0.2500
loss: 1.0282



...

Epoch 49 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-49, step-4800 ---
loss: 0.1700


Evaluation: 100%|██████████| 32/32 [00:07<00:00,  4.30it/s]

--- Eval epoch-49, step-4800 ---
accuracy: 0.8311
f1_macro: 0.6022
balanced_accuracy: 0.5842
loss: 0.4481



Evaluation: 100%|██████████| 32/32 [00:07<00:00,  4.32it/s]


### Experiment 4: Group normalization


In [9]:
model = AlzheimerCNNNormVariant(
    dataset=train_samples,
    norm_type="group",
    num_groups=8,
    init_channels=32,
    classifier_hidden_dim=128,
    dropout=0.5,
)
results.append(train_and_evaluate(model, "CNN (norm_type=group)"))



Training: CNN (norm_type=group)
AlzheimerCNNNormVariant(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 32, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block4): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3)

Epoch 0 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-0, step-96 ---
loss: 1.1142


Evaluation: 100%|██████████| 32/32 [00:02<00:00, 12.99it/s]

--- Eval epoch-0, step-96 ---
accuracy: 0.5078
f1_macro: 0.1684
balanced_accuracy: 0.2500
loss: 1.0359



...

Epoch 49 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-49, step-4800 ---
loss: 0.8499


Evaluation: 100%|██████████| 32/32 [00:02<00:00, 12.75it/s]

--- Eval epoch-49, step-4800 ---
accuracy: 0.5938
f1_macro: 0.3735
balanced_accuracy: 0.3724
loss: 0.8070



Evaluation: 100%|██████████| 32/32 [00:02<00:00, 12.62it/s]


### Experiment 5: Layer normalization


In [10]:
model = AlzheimerCNNNormVariant(
    dataset=train_samples,
    norm_type="layer",
    num_groups=8,
    init_channels=32,
    classifier_hidden_dim=128,
    dropout=0.5,
)
results.append(train_and_evaluate(model, "CNN (norm_type=layer)"))


Training: CNN (norm_type=layer)
AlzheimerCNNNormVariant(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(1, 32, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(1, 64, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(1, 128, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block4): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3)

Epoch 0 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-0, step-96 ---
loss: 1.1010


Evaluation: 100%|██████████| 32/32 [00:02<00:00, 12.27it/s]

--- Eval epoch-0, step-96 ---
accuracy: 0.5078
f1_macro: 0.1684
balanced_accuracy: 0.2500
loss: 1.0384



...

Epoch 49 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-49, step-4800 ---
loss: 0.9051


Evaluation: 100%|██████████| 32/32 [00:02<00:00, 11.85it/s]

--- Eval epoch-49, step-4800 ---
accuracy: 0.5732
f1_macro: 0.2951
balanced_accuracy: 0.3203
loss: 0.8698



Evaluation: 100%|██████████| 32/32 [00:02<00:00, 11.74it/s]


### Experiment 5: CNN + ViT hybrid


In [6]:
model = AlzheimerCNNViT(
    dataset=train_samples,
    init_channels=32,
    embed_dim=128,
    num_heads=4,
    num_transformer_layers=4,
    dropout=0.1,
)
results.append(train_and_evaluate(model, "CNN+ViT hybrid"))



Training: CNN+ViT hybrid
AlzheimerCNNViT(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (patch_embed): PatchEmbedding(
    (projection): Linear(in_features=1024, out_features=128, bias=True)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamica

Epoch 0 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-0, step-96 ---
loss: 1.1134


Evaluation: 100%|██████████| 32/32 [00:02<00:00, 13.20it/s]

--- Eval epoch-0, step-96 ---
accuracy: 0.5156
f1_macro: 0.1701
balanced_accuracy: 0.2500
loss: 1.0363



...

Epoch 49 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-49, step-4800 ---
loss: 0.0972


Evaluation: 100%|██████████| 32/32 [00:02<00:00, 15.67it/s]

--- Eval epoch-49, step-4800 ---
accuracy: 0.7344
f1_macro: 0.6318
balanced_accuracy: 0.6772
loss: 1.1015



Evaluation: 100%|██████████| 32/32 [00:02<00:00, 15.42it/s]


## 5. Results Summary

All six configurations side by side, sorted by validation accuracy.


In [7]:
df = pd.DataFrame(results)
df = df[["name", "params", "accuracy", "f1_macro", "balanced_accuracy", "loss", "train_time_seconds"]]
df = df.sort_values("accuracy", ascending=False).reset_index(drop=True)

# Format columns for readability
df["params"]   = df["params"].apply(lambda x: f"{x:,}")
df["accuracy"] = df["accuracy"].apply(lambda x: f"{x:.4f}")
df["f1_macro"] = df["f1_macro"].apply(lambda x: f"{x:.4f}")
df["balanced_accuracy"] = df["balanced_accuracy"].apply(lambda x: f"{x:.4f}")
df["loss"]     = df["loss"].apply(lambda x: f"{x:.4f}")
df["train_time_seconds"].apply(lambda x: f"{x:.4f}")

print(df.to_string(index=False))


          name  params accuracy f1_macro balanced_accuracy   loss  train_time_seconds
CNN+ViT hybrid 968,324   0.7344   0.6318            0.6772 1.1015              1416.6


## 6. Findings & Discussion

### Headline result


### Architecture finding: CNN beats CNN+ViT


### Normalization finding: Instance ≈ Group

### Normalization finding: Instance ≈ Layer 

### Practical recommendation

